In [0]:
import time
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS movie_recommender.gold

In [0]:
ratings_silver_df=spark.read.format('delta').table('movie_recommender.silver.ratings')

movies_silver_df=spark.read.format('delta').table('movie_recommender.silver.movies')

tags_silver_df=spark.read.format('delta').table('movie_recommender.silver.tags')

links_silver_df=spark.read.format('delta').table('movie_recommender.silver.links')

tmdb_silver_df=spark.read.format('delta').table('movie_recommender.silver.tmdb_metadata').where((F.col('vote_count')>0) & (F.col('vote_average')>0))

In [0]:
m=tmdb_silver_df.approxQuantile('vote_count',[0.75],0.01)[0]

C=tmdb_silver_df.select(F.avg(F.col('vote_average'))).collect()[0][0]

popular_movies_df=tmdb_silver_df.withColumn('weighted_rating',(F.col('vote_count')/(F.col('vote_count')+m))*F.col('vote_average')+(m/(F.col('vote_count')+m))*C)

popular_movies_df=popular_movies_df.where(F.col('vote_count')>=m).select(F.col('id'),F.col('title'),F.col('genres'),F.col('popularity'),F.col('vote_average'),F.col('vote_count'),F.col('weighted_rating'),F.col('release_date')).orderBy(F.col('weighted_rating').desc())

popular_movies_df.write.format('delta').mode('overwrite').saveAsTable('movie_recommender.gold.popular_movies')

In [0]:
genre_df=popular_movies_df.withColumn('genre',F.explode(F.col('genres')))

genre_stats_df=genre_df.groupBy('genre').agg({'weighted_rating':'avg','popularity':'avg','id':'count'}).withColumnRenamed("avg(weighted_rating)","avg_weighted_rating").withColumnRenamed("avg(popularity)","avg_popularity").withColumnRenamed("count(id)",'movie_count')

genre_stats_df.write.format('delta').mode('overwrite').saveAsTable('movie_recommender.gold.genre_stats')

In [0]:
genre_counts_df=popular_movies_df.join(ratings_silver_df,on=popular_movies_df.id==ratings_silver_df.movieId,how='inner').withColumn('genre',F.explode(F.col('genres'))).select(F.col('genre'),F.col('userId'),F.col('rating')).groupBy(['userId','genre']).agg({'rating':'count'}).withColumnRenamed('count(rating)','rating_count')

w=Window.partitionBy('userId').orderBy(F.col('rating_count').desc())

rating_stat_df=ratings_silver_df.groupBy('userId').agg(F.count('rating').alias('total_ratings'), F.avg('rating').alias('avg_rating'))

intermediate_df=genre_counts_df.withColumn('row_number',F.row_number().over(w)).where(F.col('row_number')==F.lit(1)).withColumnRenamed('genre','favorite_genre').select(F.col('userId'),F.col('favorite_genre'),F.col('rating_count'))

user_rating_summary_df=intermediate_df.join(rating_stat_df,on='userId').select(F.col('userId'),F.col('favorite_genre'),F.col('avg_rating'),F.col('total_ratings'))

user_rating_summary_df.write.format('delta').mode('overwrite').saveAsTable('movie_recommender.gold.user_rating_summary')